In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.sales_project;
CREATE VOLUME IF NOT EXISTS workspace.sales_project.raw_files;

In [0]:
from pyspark.sql.functions import col

raw_path = "/Volumes/workspace/sales_project/raw_files/sales_data.csv"

bronze_df = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("encoding", "ISO-8859-1")   # many Kaggle CSVs need this
    .csv(raw_path))

bronze_df.printSchema()

bronze_df.show(10,False)

bronze_df = bronze_df.select(*[col(c).alias(c.replace(" ", "_")) for c in bronze_df.columns])

bronze_df.write.mode("overwrite").saveAsTable("workspace.sales_project.bronze_sales")

display(spark.sql("DESCRIBE DETAIL workspace.sales_project.bronze_sales"))

In [0]:
from pyspark.sql import functions as F

df = spark.table("workspace.sales_project.bronze_sales")

print("Rows:", df.count())
print("Distinct rows:", df.distinct().count())

# Null counts per column
display(df.select([F.sum(F.col(f"`{c}`").isNull().cast("int")).alias(c) for c in df.columns]))

display(df.describe())